## # Mounting colab with drive|

In [2]:
# Example: reading a file
file_path = "c:\Users\Asus Vivobook\reat_eastate_project/data.csv"

import pandas as pd
df = pd.read_csv(file_path)

print(df.head())

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3995852696.py, line 2)

In [3]:

import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time


In [5]:
# Need to change as per your requirement - city name
# Match with 99acers site like for chandighars flats data site is : https://www.99acres.com/flats-in-chandigarh-ffid
# Taking value of city as 'chandigarh'
City = 'ahmedabad'

In [6]:
# User Agent
# Headers set like below:
# User Agent
headers = {
    "authority": "www.99acres.com",
    "accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "accept-language": "en-US,en;q=0.9",
    "cache-control": "no-cache",
    "dnt": "1",
    "pragma": "no-cache",
    "referer": f"https://www.99acres.com/flats-in-{City}-ffid-page",
    "sec-ch-ua": '"Chromium";v="107", "Not;A=Brand";v="8"',
    "sec-ch-ua-mobile": "?0",
    "sec-ch-ua-platform": '"Windows"',
    "sec-fetch-dest": "document",
    "sec-fetch-mode": "navigate",
    "sec-fetch-site": "same-origin",
    "sec-fetch-user": "?1",
    "upgrade-insecure-requests": "1",
    "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [11]:
import os


# Use current working directory
project_dir = os.path.join(os.getcwd())

# Define subdirectories
subdirectories = [
    "Data",
    f"Data/{City}",
    f"Data/{City}/Flats",
    f"Data/{City}/Societies",
    f"Data/{City}/Residential",
    f"Data/{City}/Independent_House"
]

# Create directories
for subdir in subdirectories:
    dir_path = os.path.join(project_dir, subdir)
    
    os.makedirs(dir_path, exist_ok=True)  # cleaner way
    print(f"Ready: {dir_path}")

Ready: C:\Users\Asus Vivobook\reat_eastate_project\Data/ahmedabad
Ready: C:\Users\Asus Vivobook\reat_eastate_project\Data/ahmedabad/Flats
Ready: C:\Users\Asus Vivobook\reat_eastate_project\Data/ahmedabad/Societies
Ready: C:\Users\Asus Vivobook\reat_eastate_project\Data/ahmedabad/Residential
Ready: C:\Users\Asus Vivobook\reat_eastate_project\Data/ahmedabad/Independent_House


In [13]:
# Put start page number and end page number.

# Page number to start extraction data
start = int(input("Enter page number where you got error in last run.\nEnter page number to start from:")) # Starting Page

# End Page number- you can change is for start i am taking 10pages at a time,
# as IPs are gettig block after some time
end = start+10

pageNumber = start
req=0

flats = pd.DataFrame()

try :
    while pageNumber < end:
        i=1
        url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
        page = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(page.content, 'html.parser')
        req+=1
        for soup in pageSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):

        # Extract property name and property sub-name
            try:
                property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
                # Extract link
                link = soup.select_one('a.srpTuple__propertyName')['href']
                society = soup.select_one('#srp_tuple_society_heading').text.strip()
            except:
                continue
            # Detail Page
            page = requests.get(link, headers=headers)
            dpageSoup = BeautifulSoup(page.content, 'html.parser')
            req += 1
            try:
                #price Range
                price = dpageSoup.select_one('#pdPrice2').text.strip()
            except:
                price = ''

            # Area
            try:
                area = soup.select_one('#srp_tuple_price_per_unit_area').text.strip()
            except:
                area =''
            # Area with Type
            try:
                areaWithType = dpageSoup.select_one('#factArea').text.strip()
            except:
                areaWithType = ''


            # Configuration
            try:
                bedRoom = dpageSoup.select_one('#bedRoomNum').text.strip()
            except:
                bedRoom = ''
            try:
                bathroom = dpageSoup.select_one('#bathroomNum').text.strip()
            except:
                bathroom = ''
            try:
                balcony = dpageSoup.select_one('#balconyNum').text.strip()
            except:
                balcony = ''

            try:
                additionalRoom = dpageSoup.select_one('#additionalRooms').text.strip()
            except:
                additionalRoom = ''


            # Address

            try:
                address = dpageSoup.select_one('#address').text.strip()
            except:
                address = ''
            # Floor Number
            try:
                floorNum = dpageSoup.select_one('#floorNumLabel').text.strip()
            except:
                floorNum = ''

            try:
                facing = dpageSoup.select_one('#facingLabel').text.strip()
            except:
                facing = ''

            try:
                agePossession = dpageSoup.select_one('#agePossessionLbl').text.strip()
            except:
                agePossession = ''

            # Nearby Locations

            try:
                nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
            except:
                nearbyLocations = ''

            # Descriptions
            try:
                description = dpageSoup.select_one('#description').text.strip()
            except:
                description = ''

            # Furnish Details
            try:
                furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#FurnishDetails').select('li')]
            except:
                furnishDetails = ''

            # Features
            if furnishDetails:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[1].select('li')]
                except:
                    features = ''
            else:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
                except:
                    features = ''



            # Rating by Features
            try:
                rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
            except:
                rating = ''
            # print(top_f)

            try:
                # Property ID
                property_id = dpageSoup.select_one('#Prop_Id').text.strip()
            except:
                property_id = ''

            # Create a dictionary with the given variables
            property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'area': area,
            'areaWithType': areaWithType,
            'bedRoom': bedRoom,
            'bathroom': bathroom,
            'balcony': balcony,
            'additionalRoom': additionalRoom,
            'address': address,
            'floorNum': floorNum,
            'facing': facing,
            'agePossession': agePossession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'furnishDetails': furnishDetails,
            'features': features,
            'rating': rating,
            'property_id': property_id
        }


            temp_df = pd.DataFrame.from_records([property_data])
            # print(temp_df)
            flats = pd.concat([flats, temp_df], ignore_index=True)
            i += 1
            # if os.path.isfile(csv_file):
            # # Append DataFrame to the existing file without header
            #     temp_df.to_csv(csv_file, mode='a', header=False, index=False)
            # else:
            #     # Write DataFrame to the file with header
            #     temp_df.to_csv(csv_file, mode='a', header=True, index=False)

            if req % 4==0:
                time.sleep(10)
            if req % 15 == 0:
                time.sleep(50)
        print(f'{pageNumber} -> {i}')
        pageNumber += 1

except AttributeError as e:
    print(e)
    print("----------------")
    print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n
            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")
    csv_file_path = f"C:/Users/Asus Vivobook/reat_eastate_project/Data/ahmedabad/Flats/flats_{City}_data-page-{start}-{pageNumber-1}.csv"

    # This file will be new every time if start page will chnage, but still taking here mode as append
    if os.path.isfile(csv_file_path):
    # Append DataFrame to the existing file without header
        flats.to_csv(csv_file_path, mode='a', header=False, index=False)
    else:
        # Write DataFrame to the file with header - first time write
        flats.to_csv(csv_file_path, mode='a', header=True, index=False)


<>:188: SyntaxWarning: invalid escape sequence '\ '
<>:188: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Asus Vivobook\AppData\Local\Temp\ipykernel_14916\413305830.py:188: SyntaxWarning: invalid escape sequence '\ '
  print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n


Enter page number where you got error in last run.
Enter page number to start from: 1


'NoneType' object has no attribute 'select'
----------------
Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.

            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.


In [13]:
# Put start page number and end page number.

# Page number to start extraction data
start = int(input("Enter page number where you got error in last run.\nEnter page number to start from:")) # Starting Page

# End Page number- you can change is for start i am taking 10pages at a time,
# as IPs are gettig block after some time
end = start+10

pageNumber = start
req=0

flats = pd.DataFrame()

try :
    while pageNumber < end:
        i=1
        url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
        page = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(page.content, 'html.parser')
        req+=1
        for soup in pageSoup.select_one('div[data-label="SEARCH"]').select('section[data-hydration-on-demand="true"]'):

        # Extract property name and property sub-name
            try:
                property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
                # Extract link
                link = soup.select_one('a.srpTuple__propertyName')['href']
                society = soup.select_one('#srp_tuple_society_heading').text.strip()
            except:
                continue
            # Detail Page
            page = requests.get(link, headers=headers)
            dpageSoup = BeautifulSoup(page.content, 'html.parser')
            req += 1
            try:
                #price Range
                price = dpageSoup.select_one('#pdPrice2').text.strip()
            except:
                price = ''

            # Area
            try:
                area = soup.select_one('#srp_tuple_price_per_unit_area').text.strip()
            except:
                area =''
            # Area with Type
            try:
                areaWithType = dpageSoup.select_one('#factArea').text.strip()
            except:
                areaWithType = ''


            # Configuration
            try:
                bedRoom = dpageSoup.select_one('#bedRoomNum').text.strip()
            except:
                bedRoom = ''
            try:
                bathroom = dpageSoup.select_one('#bathroomNum').text.strip()
            except:
                bathroom = ''
            try:
                balcony = dpageSoup.select_one('#balconyNum').text.strip()
            except:
                balcony = ''

            try:
                additionalRoom = dpageSoup.select_one('#additionalRooms').text.strip()
            except:
                additionalRoom = ''


            # Address

            try:
                address = dpageSoup.select_one('#address').text.strip()
            except:
                address = ''
            # Floor Number
            try:
                floorNum = dpageSoup.select_one('#floorNumLabel').text.strip()
            except:
                floorNum = ''

            try:
                facing = dpageSoup.select_one('#facingLabel').text.strip()
            except:
                facing = ''

            try:
                agePossession = dpageSoup.select_one('#agePossessionLbl').text.strip()
            except:
                agePossession = ''

            # Nearby Locations

            try:
                nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
            except:
                nearbyLocations = ''

            # Descriptions
            try:
                description = dpageSoup.select_one('#description').text.strip()
            except:
                description = ''

            # Furnish Details
            try:
                furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#FurnishDetails').select('li')]
            except:
                furnishDetails = ''

            # Features
            if furnishDetails:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[1].select('li')]
                except:
                    features = ''
            else:
                try:
                    features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
                except:
                    features = ''



            # Rating by Features
            try:
                rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
            except:
                rating = ''
            # print(top_f)

            try:
                # Property ID
                property_id = dpageSoup.select_one('#Prop_Id').text.strip()
            except:
                property_id = ''

            # Create a dictionary with the given variables
            property_data = {
            'property_name': property_name,
            'link': link,
            'society': society,
            'price': price,
            'area': area,
            'areaWithType': areaWithType,
            'bedRoom': bedRoom,
            'bathroom': bathroom,
            'balcony': balcony,
            'additionalRoom': additionalRoom,
            'address': address,
            'floorNum': floorNum,
            'facing': facing,
            'agePossession': agePossession,
            'nearbyLocations': nearbyLocations,
            'description': description,
            'furnishDetails': furnishDetails,
            'features': features,
            'rating': rating,
            'property_id': property_id
        }


            temp_df = pd.DataFrame.from_records([property_data])
            # print(temp_df)
            flats = pd.concat([flats, temp_df], ignore_index=True)
            i += 1
            # if os.path.isfile(csv_file):
            # # Append DataFrame to the existing file without header
            #     temp_df.to_csv(csv_file, mode='a', header=False, index=False)
            # else:
            #     # Write DataFrame to the file with header
            #     temp_df.to_csv(csv_file, mode='a', header=True, index=False)

            if req % 4==0:
                time.sleep(10)
            if req % 15 == 0:
                time.sleep(50)
        print(f'{pageNumber} -> {i}')
        pageNumber += 1

except AttributeError as e:
    print(e)
    print("----------------")
    print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n
            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.""")
    csv_file_path = f"C:/Users/Asus Vivobook/reat_eastate_project/Data/ahmedabad/Flats/flats_{City}_data-page-{start}-{pageNumber-1}.csv"

    # This file will be new every time if start page will chnage, but still taking here mode as append
    if os.path.isfile(csv_file_path):
    # Append DataFrame to the existing file without header
        flats.to_csv(csv_file_path, mode='a', header=False, index=False)
    else:
        # Write DataFrame to the file with header - first time write
        flats.to_csv(csv_file_path, mode='a', header=True, index=False)


<>:188: SyntaxWarning: invalid escape sequence '\ '
<>:188: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Asus Vivobook\AppData\Local\Temp\ipykernel_14916\413305830.py:188: SyntaxWarning: invalid escape sequence '\ '
  print("""Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.\n


Enter page number where you got error in last run.
Enter page number to start from: 1


'NoneType' object has no attribute 'select'
----------------
Your IP might have blocked. Delete Runitme and reconnect again with updating start page number.

            You would see in output above like 1 -> 15\ and so 1 is page number and 15 is data items extracted.


In [14]:
import os
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

City = "Ahmedabad"

# Input
start = int(input("Enter page number where you got error in last run.\nEnter page number to start from:"))
end = start + 10

pageNumber = start
req = 0

flats = pd.DataFrame()

try:
    while pageNumber < end:
        i = 1
        url = f'https://www.99acres.com/flats-in-{City}-ffid-page-{pageNumber}'
        
        page = requests.get(url, headers=headers)
        pageSoup = BeautifulSoup(page.content, 'html.parser')
        req += 1

        search_div = pageSoup.select_one('div[data-label="SEARCH"]')
        if not search_div:
            print(f"No data found on page {pageNumber}")
            pageNumber += 1
            continue

        for soup in search_div.select('section[data-hydration-on-demand="true"]'):

            try:
                property_name = soup.select_one('a.srpTuple__propertyName').text.strip()
                link = soup.select_one('a.srpTuple__propertyName')['href']
                society = soup.select_one('#srp_tuple_society_heading').text.strip()
            except:
                continue

            # Detail Page
            page = requests.get(link, headers=headers)
            dpageSoup = BeautifulSoup(page.content, 'html.parser')
            req += 1

            def safe_extract(selector):
                try:
                    return selector.text.strip()
                except:
                    return ''

            price = safe_extract(dpageSoup.select_one('#pdPrice2'))
            area = safe_extract(soup.select_one('#srp_tuple_price_per_unit_area'))
            areaWithType = safe_extract(dpageSoup.select_one('#factArea'))
            bedRoom = safe_extract(dpageSoup.select_one('#bedRoomNum'))
            bathroom = safe_extract(dpageSoup.select_one('#bathroomNum'))
            balcony = safe_extract(dpageSoup.select_one('#balconyNum'))
            additionalRoom = safe_extract(dpageSoup.select_one('#additionalRooms'))
            address = safe_extract(dpageSoup.select_one('#address'))
            floorNum = safe_extract(dpageSoup.select_one('#floorNumLabel'))
            facing = safe_extract(dpageSoup.select_one('#facingLabel'))
            agePossession = safe_extract(dpageSoup.select_one('#agePossessionLbl'))

            try:
                nearbyLocations = [i.text.strip() for i in dpageSoup.select_one('div.NearByLocation__tagWrap').select('span.NearByLocation__infoText')]
            except:
                nearbyLocations = ''

            description = safe_extract(dpageSoup.select_one('#description'))

            try:
                furnishDetails = [i.text.strip() for i in dpageSoup.select_one('#FurnishDetails').select('li')]
            except:
                furnishDetails = ''

            try:
                features = [i.text.strip() for i in dpageSoup.select('#features')[0].select('li')]
            except:
                features = ''

            try:
                rating = [i.text for i in dpageSoup.select_one('div.review__rightSide>div>ul>li>div').select('div.ratingByFeature__circleWrap')]
            except:
                rating = ''

            property_id = safe_extract(dpageSoup.select_one('#Prop_Id'))

            property_data = {
                'property_name': property_name,
                'link': link,
                'society': society,
                'price': price,
                'area': area,
                'areaWithType': areaWithType,
                'bedRoom': bedRoom,
                'bathroom': bathroom,
                'balcony': balcony,
                'additionalRoom': additionalRoom,
                'address': address,
                'floorNum': floorNum,
                'facing': facing,
                'agePossession': agePossession,
                'nearbyLocations': nearbyLocations,
                'description': description,
                'furnishDetails': furnishDetails,
                'features': features,
                'rating': rating,
                'property_id': property_id
            }

            temp_df = pd.DataFrame([property_data])
            flats = pd.concat([flats, temp_df], ignore_index=True)

            i += 1

            # Delay to avoid blocking
            if req % 4 == 0:
                time.sleep(10)
            if req % 15 == 0:
                time.sleep(50)

        print(f'{pageNumber} -> {i}')
        pageNumber += 1

except AttributeError as e:
    print("Error:", e)
    print("Your IP might be blocked. Restart and continue from last page.")

# ✅ SAVE DATA (Jupyter path)
folder_path = os.path.join("Data", City, "Flats")
os.makedirs(folder_path, exist_ok=True)

csv_file_path = os.path.join(
    folder_path,
    f"flats_{City}_data_page_{start}_{pageNumber-1}.csv"
)

if os.path.isfile(csv_file_path):
    flats.to_csv(csv_file_path, mode='a', header=False, index=False)
else:
    flats.to_csv(csv_file_path, mode='w', header=True, index=False)

print("✅ Data saved at:", csv_file_path)

Enter page number where you got error in last run.
Enter page number to start from: 0


No data found on page 0
No data found on page 1
No data found on page 2
No data found on page 3
No data found on page 4
No data found on page 5
No data found on page 6
No data found on page 7
No data found on page 8
No data found on page 9
✅ Data saved at: Data\Ahmedabad\Flats\flats_Ahmedabad_data_page_0_9.csv
